## Challenge 1: Byte Alchemist

File: challenge1.txt

Tugas:
- hex → bytes
- bytes → base64
- base64 → bytes
- bytes → int
- int → bytes
- pastikan hasil akhir identik

In [ ]:
import hashlib
from pathlib import Path
from base64 import b64encode, b64decode
from Crypto.Util.number import long_to_bytes

In [ ]:
# read the buffer
p = Path("/mnt/d/my-kisah/crypto/1.cryptopals/set_1/py_chall/")
f = p / "challenge1.txt"

# raw hex
h = f.read_text()

# hex -> bytes
bt = bytes.fromhex(h)

# bytes -> base64
b64 = b64encode(bt)

# base64 -> bytes
bb64 = b64decode(b64)

# bytes -> int
num = int.from_bytes(bb64, "big")

# int -> bytes
byte_num = long_to_bytes(num)

# turn byte_num into digest sha256
m = hashlib.sha256()
m.update(byte_num)
ct = m.hexdigest()

# print-out
print(f"hex form           :   {h}")
print(f"bytes form         :   {bt}")
print(f"base64 form        :   {b64}")
print(f"bytes form         :   {bb64}")
print(f"integer form       :   {num}")
print(f"byte integer form  :   {byte_num}")
print(f"sha256 form        :   {ct}")

### Validation

In [ ]:
answer = ct
sha256 = "02f9d39e787d2dd2536beeb8b1711cc92b58d3a7ab9fa84f0af718cce6f7d0ff"

if answer == sha256:
    print("correct...")
else:
    print("try again...")

## Challenge 2: XOR Twins

Referensi: Sesi 2 (Fixed XOR) + Sesi 3 (Single-byte XOR cipher)

Tugas:

1. Buat fungsi fixed_xor(buf1: bytes, buf2: bytes) -> bytes — XOR dua buffer panjang sama, byte-per-byte. Kalau panjangnya beda, raise ValueError.
2. Buat fungsi single_byte_xor(data: bytes, key: int) -> bytes — XOR seluruh data dengan satu key byte (0–255) berulang.
3. Buat fungsi scorer score_english(text: bytes) -> float — pakai frequency analysis (ETAOIN SHRDLU) buat nilai seberapa "mirip English" hasil decode-nya. Bebas metode: character frequency table, atau itung rasio printable ASCII, dsb.
4. Buat fungsi crack_single_byte_xor(ciphertext: bytes) -> tuple[int, bytes, float] — brute-force 256 key, return (key, plaintext, score) terbaik.

In [93]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))
    
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    ascii = [i for i in range(32,127)]
    letters = [69, 84, 65, 79, 73, 78, 83, 72, 82, 68, 76, 67, 85, 101, 116, 97, 111, 105, 110, 115, 104, 114, 100, 108, 99, 117, 32]
    score = 0

    for t in text:
        if t not in ascii:
            return 0
        if t in letters:
            match t:
                case 69 | 97:
                    score += 4
                case 84 | 111:
                    score += 3
                case 65 | 105:
                    score += 2
                case 32 | 110:
                    score += 2
            score += 1
    return score/len(text)

def crack_single_byte_xor(ciphertext: bytes):
    cand = []

    for k in range(256):
        p = b""
        for c in ciphertext:
            p += (c ^ k).to_bytes(1,"big")

        score = score_english(p)
        if k == 0:
            cand.append(k)
            cand.append(p)
            cand.append(score)
            continue

        if score < cand[2]:
            continue
        else:
            cand[0] = k
            cand[1] = p
            cand[2] = score
    
    return tuple(cand)

In [98]:
ciphertext_hex = "1b37373331363f78151b7f2b783431333d78397828372d363c78373e783a393b3736"
ct = bytes.fromhex(ciphertext_hex)

cand = crack_single_byte_xor(ct)
key = cand[0]
pt = cand[1]
score = cand[2]

print(f"ciphertext  : {ct}")
print(f"key         : {key}")
print(f"plaintext   : {pt.decode()}")
print(f"score       : {score}")

print("\nTesting Self-XOR:")
zeroed = fixed_xor(pt, pt)
assert zeroed == b"\x00" * len(pt), "fixed_xor lo salah, bro"
print(zeroed)  # harus print b'\x00\x00\x00...\x00' sepanjang plaintext

ciphertext  : b'\x1b77316?x\x15\x1b\x7f+x413=x9x(7-6<x7>x:9;76'
key         : 88
plaintext   : Cooking MC's like a pound of bacon
score       : 2.088235294117647

Testing Self-XOR:
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


## Challenge 3: The Needle in the Haystack
